In [2]:
# imports
import pandas as pd # Manejo de datos en tablas
import numpy as np # Operaciones numéricas
import matplotlib.pyplot as plt  # Gráficas
import seaborn as sns  # Gráficas más avanzadas
from sklearn.experimental import enable_iterative_imputer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.compose import ColumnTransformer
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LassoCV

In [3]:
# Reading
df = pd.read_csv("PRSA_data_2010.1.1-2014.12.31.csv")

df = df.rename(columns={"No": "NO",
                        "year": "YEAR",
                        "month": "MONTH",
                        "day": "DAY",
                        "hour": "HOUR",
                        "pm2.5": "PM25",
                        "DEWP": "DEWPOINT",
                        "TEMP": "TEMP",
                        "PRES": "PRESION",
                        "cbwd": "WIND_DIRECTION",
                        "Iws": "WIND_SPEED",
                        "Is": "SNOW",
                        "Ir": "RAIN"})
df.info()  # Revisar el tipos de datos y columnas presentes

<class 'pandas.DataFrame'>
RangeIndex: 43824 entries, 0 to 43823
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   NO              43824 non-null  int64  
 1   YEAR            43824 non-null  int64  
 2   MONTH           43824 non-null  int64  
 3   DAY             43824 non-null  int64  
 4   HOUR            43824 non-null  int64  
 5   PM25            41757 non-null  float64
 6   DEWPOINT        43824 non-null  int64  
 7   TEMP            43824 non-null  float64
 8   PRESION         43824 non-null  float64
 9   WIND_DIRECTION  43824 non-null  str    
 10  WIND_SPEED      43824 non-null  float64
 11  SNOW            43824 non-null  int64  
 12  RAIN            43824 non-null  int64  
dtypes: float64(4), int64(8), str(1)
memory usage: 4.3 MB


In [5]:
# Cleaning and Lags

# ==========================================
# 1. Lags del Target (PM2.5)
# ==========================================
df['PM25_lag1'] = df['PM25'].shift(1)
df['PM25_lag2'] = df['PM25'].shift(2)
df['PM25_lag24'] = df['PM25'].shift(24) # El ciclo de ayer a la misma hora

# ==========================================
# 2. Lags inmediatos del clima (Pasado reciente)
# ==========================================
variables_clima = ['DEWPOINT', 'TEMP', 'PRESION', 'WIND_SPEED']

for var in variables_clima:
    df[f'{var}_lag1'] = df[var].shift(1)
    df[f'{var}_lag2'] = df[var].shift(2)

# ==========================================
# 3. El Efecto Acumulado (Ventanas Móviles / Rolling Windows)
# ==========================================
# Calculamos el promedio de las últimas 6 horas para capturar la tendencia sostenida
for var in variables_clima:
    # rolling(window=6) agrupa las últimas 6 filas, .mean() saca el promedio
    df[f'{var}_media_6h'] = df[var].rolling(window=6).mean()

# (Opcional) Si creen que algo más largo importa, pueden hacer una de 12 horas
for var in variables_clima:
    df[f'{var}_media_12h'] = df[var].rolling(window=12).mean()

# ==========================================
# 4. Limpieza final
# ==========================================
# Como el lag 24 y la ventana de 12 horas necesitan datos del pasado, 
# las primeras 24 filas de todo el dataframe quedarán con NaN. Las borramos.
df_final = df.dropna().reset_index(drop=True)
display(df_final.info())

<class 'pandas.DataFrame'>
RangeIndex: 40390 entries, 0 to 40389
Data columns (total 32 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   NO                    40390 non-null  int64  
 1   YEAR                  40390 non-null  int64  
 2   MONTH                 40390 non-null  int64  
 3   DAY                   40390 non-null  int64  
 4   HOUR                  40390 non-null  int64  
 5   PM25                  40390 non-null  float64
 6   DEWPOINT              40390 non-null  int64  
 7   TEMP                  40390 non-null  float64
 8   PRESION               40390 non-null  float64
 9   WIND_DIRECTION        40390 non-null  str    
 10  WIND_SPEED            40390 non-null  float64
 11  SNOW                  40390 non-null  int64  
 12  RAIN                  40390 non-null  int64  
 13  PM25_lag1             40390 non-null  float64
 14  PM25_lag2             40390 non-null  float64
 15  PM25_lag24            40390 no

None

In [7]:
# Scaling and Imputing

# ==========================================
# 1. DEFINIR LOS PIPELINES POR SEPARADO
# ==========================================

# Pipeline para Números: Primero imputa con la mediana, luego escala.
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Pipeline para Texto: Primero imputa con el más frecuente, luego hace OneHotEncoding.
# handle_unknown='ignore' evita que el modelo falle si en el futuro aparece una dirección de viento nueva.
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# ==========================================
# 2. COMBINARLOS EN EL COLUMN TRANSFORMER
# ==========================================
# make_column_selector busca automáticamente las columnas según su tipo (dtype)
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline, make_column_selector(dtype_include=['float64', 'int64'])),
    ('cat', categorical_pipeline, make_column_selector(dtype_include=['object', 'category']))
], remainder='passthrough') # 'passthrough' por si hay alguna columna que no queramos tocar


# ==========================================
# 3. EL GRAN TRUCO: EL PIPELINE GLOBAL
# ==========================================
# Ahora unimos el preprocesador directamente con tu modelo Lasso. 
# ¡Así todo se ejecuta de golpe de forma segura!
modelo_final_lasso = Pipeline(steps=[
    ('preprocesamiento', preprocessor),
    ('modelo_lasso', LassoCV(cv=5, random_state=42)) # Idealmente con TimeSeriesSplit
])

# ==========================================
# 4. ENTRENAMIENTO SECUENCIAL (Tu Train/Test Split)
# ==========================================
# Recuerda separar tus datos respetando el tiempo (como vimos antes)
corte = int(len(df_final) * 0.8)
X_train = df_final.iloc[:corte].drop(columns=['PM25'])
y_train = df_final.iloc[:corte]['PM25']

X_test = df_final.iloc[corte:].drop(columns=['PM25'])
y_test = df_final.iloc[corte:]['PM25']

# Al hacer .fit(), el Pipeline ejecuta la imputación y el escalado de los números,
# el OneHotEncoder de las categorías, junta todo en una matriz pesada interna,
# y se la entrega directamente a Lasso. ¡Todo en una sola línea!
modelo_final_lasso.fit(X_train, y_train)


¡Pipeline entrenado con éxito con ColumnTransformer!


In [ ]:
# Lasso



In [ ]:
# Ridge

